In [7]:
# Import required libs
import numpy as np
import pandas as pd
import os
import json
from dotenv import load_dotenv
from mistralai import Mistral

In [2]:
# Reading Input from Json file
with open('documents.json', 'rt', encoding='utf-8') as f_out:
    raw_docs = json.load(f_out)

In [4]:
# Preparation of list data new form
documents = []

for course_dict in raw_docs:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        documents.append(doc)

In [112]:
# Creating LLM connection

load_dotenv()
api_key = os.getenv('MISTRAL_API_KEY_2')

# Create LLM Client
client = Mistral(api_key=api_key)
model = 'devstral-small-2507'

In [113]:
# Creating LLM Function

def llm(prompt):
    try:
        response = client.chat.complete(
            model = model,
            messages = [
                {
                    'role': 'user',
                    'content': prompt
                }
            ]
        )
        answer = response.choices[0].message.content
        return answer
    except Exception as e:
        print(f"An error occured: {e}")

In [50]:
# Build Prompt Function

prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION

<QUESTION>
{question}
</QUESTION>

<CONTEXT
{context}
</CONTEXT>
""".strip()

def build_prompt(query, search_results):
    context = ""
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"

    prompt = prompt_template.format(question=query, context=context)
    return prompt

In [51]:
# Testing with minsearch
from minsearch.append import AppendableIndex

In [52]:
index = AppendableIndex(
    text_fields = ['question', 'text', 'section'],
    keyword_fields = ['course']
)

index.fit(documents)

In [53]:
def search(query):
    boost = {'question':3.0, 'section':0.5}
    results = index.search(
        query = query,
        filter_dict = {'course': 'data-engineering-zoomcamp'},
        boost_dict = boost,
        num_results = 5,
        output_ids = True
    )
    return results

In [59]:
# RAG Functions
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [54]:
query = 'How Can i run kafka?'
res = search(query)

In [55]:
prompt = build_prompt(query, res)

### --- Agentic Rag ---

In [106]:
prompt_template = """
You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.
At the beginning the context is EMPTY.

<QUESTION>
{question}
</QUESTION>

<CONTEXT> 
{context}
</CONTEXT>

If CONTEXT is EMPTY, you can use our FAQ database. Please answer with the following shown JSON file format only.
In this case, use the following output template:

{{
"action": "SEARCH",
"reasoning": "<add your reasoning here>"
}}

If you can answer the QUESTION using CONTEXT, use this template:

{{
"action": "ANSWER",
"answer": "<your answer>",
"source": "CONTEXT"
}}

If the context doesn't contain the answer, use your own knowledge to answer the question

{{
"action": "ANSWER",
"answer": "<your answer>",
"source": "OWN_KNOWLEDGE"
}}

Please return only with the JSON file only. You don't have to defined it is JSON.
""".strip()

In [119]:
def build_prompt(search_results):
    context = ""
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
        
    return context.strip()

In [124]:
question = "Can I still Join the course?"
context = 'EMPTY'

In [125]:
prompt = prompt_template.format(question=question, context=context)

In [126]:
answer_json = llm(prompt)

In [127]:
answer = json.loads(answer_json)

In [128]:
answer['action']

'SEARCH'

In [129]:
search_results = search(question)
context = build_prompt(search_results)
prompt = prompt_template.format(question=question, context=context)
print(prompt)

You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.
At the beginning the context is EMPTY.

<QUESTION>
Can I still Join the course?
</QUESTION>

<CONTEXT> 
section: General course-related questions
question: Course - Can I still join the course after the start date?
answer: Yes, even if you don't register, you're still eligible to submit the homeworks.
Be aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.

section: General course-related questions
question: Certificate - Can I follow the course in a self-paced mode and get a certificate?
answer: No, you can only get a certificate if you finish the course with a “live” cohort. We don't award certificates for the self-paced mode. The reason is you need to peer-review capstone(s) after submitting a project. You can only peer-review projects at the time the cour

In [130]:
answer_json = llm(prompt)

In [131]:
print(answer_json)

{
"action": "ANSWER",
"answer": "Yes, you can still join the course after the start date. You don't need to register to be eligible to submit the homeworks, but be aware of the deadlines for turning in the final projects.",
"source": "CONTEXT"
}


In [133]:
prompt_template = """
You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.

The CONTEXT is build with the documents from our FAQ database.
SEARCH_QUERIES contains the queries that were used to retrieve the documents
from FAQ to and add them to the context.
PREVIOUS_ACTIONS contains the actions you already performed.

At the beginning the CONTEXT is empty.

You can perform the following actions:

- Search in the FAQ database to get more data for the CONTEXT
- Answer the question using the CONTEXT
- Answer the question using your own knowledge

For the SEARCH action, build search requests based on the CONTEXT and the QUESTION.
Carefully analyze the CONTEXT and generate the requests to deeply explore the topic. 

Don't use search queries used at the previous iterations.

Don't repeat previously performed actions.

Don't perform more than {max_iterations} iterations for a given student question.
The current iteration number: {iteration_number}. If we exceed the allowed number 
of iterations, give the best possible answer with the provided information.

Output templates:

If you want to perform search, use this template:

{{
"action": "SEARCH",
"reasoning": "<add your reasoning here>",
"keywords": ["search query 1", "search query 2", ...]
}}

If you can answer the QUESTION using CONTEXT, use this template:

{{
"action": "ANSWER_CONTEXT",
"answer": "<your answer>",
"source": "CONTEXT"
}}

If the context doesn't contain the answer, use your own knowledge to answer the question

{{
"action": "ANSWER",
"answer": "<your answer>",
"source": "OWN_KNOWLEDGE"
}}

<QUESTION>
{question}
</QUESTION>

<SEARCH_QUERIES>
{search_queries}
</SEARCH_QUERIES>

<CONTEXT> 
{context}
</CONTEXT>

<PREVIOUS_ACTIONS>
{previous_actions}
</PREVIOUS_ACTIONS>
""".strip()

In [158]:
def build_context(search_results):
    context = ""
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
        
    return context.strip()

In [134]:
### Agentic Search

question = 'how do I do well on Module 1'
max_iterations = 3
iteration_number = 0
search_queries = []
search_results = []
previous_actions = []

In [182]:
context = build_context(search_results)
prompt = prompt_template.format(
    question=question,
    context=context,
    search_queries="\n".join(search_queries),
    previous_actions='\n'.join([json.dumps(a) for a in previous_actions]),
    max_iterations=3,
    iteration_number=1
)

In [183]:
answer_json = llm(prompt)

In [208]:
answer = json.loads(answer_json)

In [209]:
previous_actions.append(answer)

In [199]:
keywords = answer['keywords']

KeyError: 'keywords'

In [210]:
for kw in keywords:
    search_queries.append(kw)
    sr = search(kw)
    search_results.extend(sr)

In [152]:
# Remove duplicate
def dedup(seq):
    seen = set()
    result = []
    for el in seq:
        _id = el['_id']
        if _id in seen:
            continue
        seen.add(_id)
        result.append(el)
    return result

search_results = dedup(search_results)

In [153]:
search_results = dedup(search_results)

In [184]:
len(search_results)

7

In [201]:
# itr - 1
iteration_number = 2
context = build_context(search_results)

prompt = prompt_template.format(
    question=question,
    context=context,
    search_queries="\n".join(search_queries),
    previous_actions='\n'.join([json.dumps(a) for a in previous_actions]),
    max_iterations=3,
    iteration_number=1
)

In [202]:
answer_json = llm(prompt)

In [203]:
print(answer_json)

{
"action": "ANSWER",
"answer": "To do well on Module 1, you should focus on understanding the core concepts of Docker and Terraform, as these are the primary topics covered in this module. Make sure to follow the course instructions carefully, especially when setting up your environment. Additionally, practice hands-on exercises and projects to gain practical experience. If you encounter any issues, such as module not found errors, refer to the FAQs for solutions. For example, if you face a 'ModuleNotFoundError: No module named 'psycopg2'' error, you can resolve it by installing the psycopg2-binary package using pip. Regularly review the course materials and seek help from the course community or instructor if needed.",
"source": "OWN_KNOWLEDGE"
}


In [211]:
print(answer['answer'])

To be successful in Module 1, you should focus on understanding the core concepts of Docker and Terraform. Here are some tips to help you succeed:

1. **Understand the Basics**: Make sure you have a solid understanding of Docker and Terraform fundamentals. This includes knowing how to create and manage Docker containers and how to write Terraform configuration files.

2. **Practice Hands-On**: Apply what you learn by practicing with real-world examples. Set up your own Docker containers and write Terraform scripts to automate infrastructure provisioning.

3. **Utilize Resources**: Use the resources provided in the module, such as documentation, tutorials, and example projects. These can help you grasp the concepts more effectively.

4. **Participate in Discussions**: Engage in course forums or discussion groups to ask questions and share knowledge with fellow students. This can provide different perspectives and solutions to problems you might encounter.

5. **Review Assessment Require

In [205]:
question = "what do I need to do to be successful at module 1?"

search_queries = []
search_results = []
previous_actions = []


iteration = 0

while True:
    print(f'ITERATION #{iteration}...')

    context = build_context(search_results)
    prompt = prompt_template.format(
        question=question,
        context=context,
        search_queries="\n".join(search_queries),
        previous_actions='\n'.join([json.dumps(a) for a in previous_actions]),
        max_iterations=3,
        iteration_number=iteration
    )

    print(prompt)

    answer_json = llm(prompt)
    answer = json.loads(answer_json)
    print(json.dumps(answer, indent=2))

    previous_actions.append(answer)

    action = answer['action']
    if action != 'SEARCH':
        break

    keywords = answer['keywords']
    search_queries = list(set(search_queries) | set(keywords))
    
    for k in keywords:
        res = search(k)
        search_results.extend(res)

    search_results = dedup(search_results)
    
    iteration = iteration + 1
    if iteration >= 4:
        break

    print()

ITERATION #0...
You're a course teaching assistant.

You're given a QUESTION from a course student and that you need to answer with your own knowledge and provided CONTEXT.

The CONTEXT is build with the documents from our FAQ database.
SEARCH_QUERIES contains the queries that were used to retrieve the documents
from FAQ to and add them to the context.
PREVIOUS_ACTIONS contains the actions you already performed.

At the beginning the CONTEXT is empty.

You can perform the following actions:

- Search in the FAQ database to get more data for the CONTEXT
- Answer the question using the CONTEXT
- Answer the question using your own knowledge

For the SEARCH action, build search requests based on the CONTEXT and the QUESTION.
Carefully analyze the CONTEXT and generate the requests to deeply explore the topic. 

Don't use search queries used at the previous iterations.

Don't repeat previously performed actions.

Don't perform more than 3 iterations for a given student question.
The current 

In [207]:
print(answer['answer'])

To be successful in Module 1, you should focus on understanding the core concepts of Docker and Terraform. Here are some tips to help you succeed:

1. **Understand the Basics**: Make sure you have a solid understanding of Docker and Terraform fundamentals. This includes knowing how to create and manage Docker containers and how to write Terraform configuration files.

2. **Practice Hands-On**: Apply what you learn by practicing with real-world examples. Set up your own Docker containers and write Terraform scripts to automate infrastructure provisioning.

3. **Utilize Resources**: Use the resources provided in the module, such as documentation, tutorials, and example projects. These can help you grasp the concepts more effectively.

4. **Participate in Discussions**: Engage in course forums or discussion groups to ask questions and share knowledge with fellow students. This can provide different perspectives and solutions to problems you might encounter.

5. **Review Assessment Require

In [212]:
iteration

1

## --- Function Calling ('Tool Use') ---

In [215]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5,
        output_ids=True
    )

    return results

In [216]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [226]:
# Creating Open Router Clients
from openai import OpenAI

load_dotenv()

openrouter_api_key = os.getenv('OPENROUTER_API_EKY')

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=openrouter_api_key,
)

In [227]:
completion = client.chat.completions.create(
  extra_body={},
  model="openai/gpt-4o-mini",
  messages=[
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": "What is in this image?"
        },
        {
          "type": "image_url",
          "image_url": {
            "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/d/dd/Gfp-wisconsin-madison-the-nature-boardwalk.jpg/2560px-Gfp-wisconsin-madison-the-nature-boardwalk.jpg"
          }
        }
      ]
    }
  ]
)
print(completion.choices[0].message.content)

The image depicts a pathway made of wooden boards winding through a lush green landscape. The path is surrounded by tall grass and vegetation, and the background features a blue sky with some clouds. This scene appears to be a tranquil natural setting, likely a marsh or wetland area.


In [ ]:
question = "How do I do well in module 1?"

developer_prompt = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.
""".strip()

tools = [search_tool]

chat_messages = [
    {"role": "developer", "content": developer_prompt},
    {"role": "user", "content": question}
]

response = client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=tools
)
response.output